In [ ]:
%load_ext autoreload
%autoreload 2
%env ANYWIDGET_HMR=1

import celldega as dega
import pandas as pd
import scanpy as sc
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import tifffile as tiff
import os

dega.__version__

In [ ]:
def make_column_names_unique_fast(df):
    counts = defaultdict(int)
    used = set()
    new_cols = []

    for col in df.columns:
        if col not in used:
            new_cols.append(col)
            used.add(col)
            counts[col] += 1
        else:
            while True:
                new_name = f"{col}_{counts[col]}"
                counts[col] += 1
                if new_name not in used:
                    new_cols.append(new_name)
                    used.add(new_name)
                    break

    df.columns = new_cols
    return df

In [ ]:
# data/mouse.cortex.242genes.100um/transcripts.csv:

# data/mouse.cortex.242genes.100um/transcripts_per_feature.csv:

# data/mouse.cortex.242genes.100um/feature_metadata.csv:

In [ ]:
raw_cells_cortex = pd.read_csv("../../scverse-owkin_thick_merfish/dryad-2024-08-14/data/mouse.cortex.242genes.100um/feature_metadata.csv")

raw_cells_cortex["geometry"] = [list(x) for x in zip(raw_cells_cortex["center_x"], 
                                                    raw_cells_cortex["center_y"], 
                                                    raw_cells_cortex["center_z"])]

raw_cells_cortex['name'] = raw_cells_cortex['id']
raw_cells_cortex['batch'] = 0
raw_cells_cortex.set_index("id", inplace=True)

raw_cells_cortex_small = raw_cells_cortex[["geometry", "name", "batch"]]

## original landscape files (1rep)

In [ ]:
os.makedirs("data/landscape_files/point_cloud_100um_mouse_cortex/", exist_ok=True)
raw_cells_cortex_small.to_parquet("data/landscape_files/point_cloud_100um_mouse_cortex/cell_metadata.parquet")

In [ ]:
cbg = pd.read_csv("../../scverse-owkin_thick_merfish/dryad-2024-08-14/data/mouse.cortex.242genes.100um/transcripts_per_feature.csv", index_col=0)

if cbg.columns.duplicated().any():
    print("Duplicate columns found in CBG matrix. Making column names unique.")
    cbg = make_column_names_unique_fast(cbg)

In [ ]:
dega.pre.make_meta_gene(cbg, "data/landscape_files/point_cloud_100um_mouse_cortex/meta_gene.parquet")
dega.pre.save_cbg_gene_parquets("MERSCOPE", "data/landscape_files/point_cloud_100um_mouse_cortex", cbg, verbose=True)

os.makedirs("data/landscape_files/point_cloud_100um_mouse_cortex/cell_clusters", exist_ok=True)

raw_cells_cortex_small['cluster'] = "0"

cluster = raw_cells_cortex_small[["cluster"]]

cluster.to_parquet("data/landscape_files/point_cloud_100um_mouse_cortex/cell_clusters/cluster.parquet")

meta_cluster = pd.DataFrame(index=["0"])
meta_cluster.loc["0", "color"] = "#1f77b4"
meta_cluster.loc["0", "count"] = len(cluster)
meta_cluster.to_parquet("data/landscape_files/point_cloud_100um_mouse_cortex/cell_clusters/meta_cluster.parquet")

path_landscape_files = "data/landscape_files/point_cloud_100um_mouse_cortex"
base_url = f"http://localhost:{dega.viz.get_local_server()}/{path_landscape_files}"

## 80 reps

In [ ]:
df = raw_cells_cortex_small.copy()

n_repeats = 80
z_gap = 100

# geometry is [x, y, z]
geom = np.vstack(df["geometry"].to_numpy())

x = geom[:, 0]
y = geom[:, 1]
z = geom[:, 2]

base_z_max = z.max()

n_cells = len(df)
n_new = n_cells * n_repeats

# repeat original x/y 60 times
new_x = np.tile(x, n_repeats)
new_y = np.tile(y, n_repeats)

# place each duplicated layer above the previous highest z
# layer 1: max_z + 100 + original z offset
# layer 2: max_z + 200 + original z offset, etc.
z_offsets = np.repeat(np.arange(1, n_repeats + 1) * z_gap, n_cells)
new_z = np.tile(z, n_repeats) + base_z_max + z_offsets

new_geoms = list(map(list, np.column_stack([new_x, new_y, new_z])))

new_ids = [f"cell-{i}" for i in range(1, n_new + 1)]

df_new = pd.DataFrame(
    {
        "geometry": new_geoms,
        "name": new_ids,
        "batch": 0,
    },
    index=new_ids,
)

expanded_cells = pd.concat([df, df_new], axis=0)
expanded_cells

In [ ]:
os.makedirs("data/landscape_files/point_cloud_100um_mouse_cortex_80reps/", exist_ok=True)
expanded_cells.to_parquet("data/landscape_files/point_cloud_100um_mouse_cortex_80reps/cell_metadata.parquet")

In [ ]:
cbg = pd.read_csv("../../scverse-owkin_thick_merfish/dryad-2024-08-14/data/mouse.cortex.242genes.100um/transcripts_per_feature.csv", index_col=0)
cbg

In [ ]:
# original cell x gene matrix
expr = cbg.copy()

# new_ids should be the same IDs used in df_new / expanded_cells
# e.g. ["cell-1", "cell-2", ...]

zero_expr = pd.DataFrame(
    0,
    index=new_ids,
    columns=expr.columns,
    dtype=expr.dtypes.iloc[0] if expr.dtypes.nunique() == 1 else float,
)

expanded_expr = pd.concat([expr, zero_expr], axis=0)
expanded_expr

In [ ]:
if expanded_expr.columns.duplicated().any():
    print("Duplicate columns found in CBG matrix. Making column names unique.")
    expanded_expr = make_column_names_unique_fast(expanded_expr)

In [ ]:
dega.pre.make_meta_gene(expanded_expr, "data/landscape_files/point_cloud_100um_mouse_cortex_80reps/meta_gene.parquet")

In [ ]:
dega.pre.save_cbg_gene_parquets("MERSCOPE", "data/landscape_files/point_cloud_100um_mouse_cortex_80reps", expanded_expr, verbose=True)

In [ ]:
os.makedirs("data/landscape_files/point_cloud_100um_mouse_cortex_80reps/cell_clusters", exist_ok=True)

In [ ]:
expanded_cells['cluster'] = "0"
cluster = expanded_cells[["cluster"]]
cluster.to_parquet("data/landscape_files/point_cloud_100um_mouse_cortex_80reps/cell_clusters/cluster.parquet")

In [ ]:
meta_cluster = pd.DataFrame(index=["0"])
meta_cluster.loc["0", "color"] = "#1f77b4"
meta_cluster.loc["0", "count"] = len(cluster)
meta_cluster.to_parquet("data/landscape_files/point_cloud_100um_mouse_cortex_80reps/cell_clusters/meta_cluster.parquet")

In [ ]:
path_landscape_files = "data/landscape_files/point_cloud_100um_mouse_cortex_80reps"
base_url = f"http://localhost:{dega.viz.get_local_server()}/{path_landscape_files}"

In [ ]:
landscape = dega.viz.Landscape(
    technology='point-cloud', 
    height=600,
    base_url=base_url,
    rotation_x=90
)

landscape